<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [3]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [4]:
# TODO: implementa vectorizar(texto, vocabulario)
import numpy as np
x = np.array([1, 1, 1, 1, 1])

In [5]:
corpus = [
    ("spam", "gana dinero gratis"),
    ("spam", "dinero gratis ahora"),
    ("spam", "premio dinero gratis"),
    ("spam", "gana premio ahora"),
    ("spam", "en la reunion habra dinero gratis"),
    ("normal", "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]


def vectorizar(texto, vocabulario):
    palabras = texto.lower().split()

    vector = []

    for palabra in vocabulario:
        if palabra in palabras:
            vector.append(1)
        else:
            vector.append(0)

    return vector


# Vectorizamos todos los correos
X = []

for etiqueta, texto in corpus:
    vector = vectorizar(texto, vocabulario)
    X.append(vector)


# Mostramos los resultados
for (etiqueta, texto), vector in zip(corpus, X):
    print(etiqueta, "|", texto, "->", vector)

spam | gana dinero gratis -> [1, 1, 0, 0, 0]
spam | dinero gratis ahora -> [1, 1, 0, 0, 0]
spam | premio dinero gratis -> [1, 1, 1, 0, 0]
spam | gana premio ahora -> [0, 0, 1, 0, 0]
spam | en la reunion habra dinero gratis -> [1, 1, 0, 0, 1]
normal | daremos un premio despues de la reunion -> [0, 0, 1, 0, 1]
normal | reunion de proyecto -> [0, 0, 0, 1, 1]
normal | proyecto para mañana -> [0, 0, 0, 1, 0]
normal | reunion mañana -> [0, 0, 0, 0, 1]
normal | informe del proyecto -> [0, 0, 0, 1, 0]
normal | informe del proyecto -> [0, 0, 0, 1, 0]


## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [13]:
# TODO: calcula P(Y=spam) y P(Y=normal)
total_correos = len(corpus)

spam = 0
normal = 0

for etiqueta, texto in corpus:
    if etiqueta == "spam":
        spam += 1
    else:
        normal += 1

P_spam = spam / total_correos
P_normal = normal / total_correos

print("P(spam) =", P_spam)
print("P(normal) =", P_normal)


P(spam) = 0.45454545454545453
P(normal) = 0.5454545454545454
6


## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [11]:
# ============================================================
# PARTE 3: Calcular P(X_i = 1 | Y)
# ============================================================

# Separamos los vectores según su clase
spam_vectores = []
normal_vectores = []

for etiqueta, texto in corpus:

    # Convertimos el correo en su vector binario
    vector = vectorizar(texto, vocabulario)

    # Guardamos el vector dependiendo de su clase
    if etiqueta == "spam":
        spam_vectores.append(vector)
    else:
        normal_vectores.append(vector)


# ------------------------------------------------------------
# Calculamos P(X_i = 1 | spam) y P(X_i = 1 | normal)
# ------------------------------------------------------------

prob_spam = []
prob_normal = []

for i, palabra in enumerate(vocabulario):

    # Número de correos spam donde aparece la palabra i
    conteo_spam = sum(vector[i] for vector in spam_vectores)

    # Número de correos normales donde aparece la palabra i
    conteo_normal = sum(vector[i] for vector in normal_vectores)

    # Probabilidades condicionales
    p_spam = conteo_spam / len(spam_vectores)
    p_normal = conteo_normal / len(normal_vectores)

    # Guardamos las probabilidades
    prob_spam.append(p_spam)
    prob_normal.append(p_normal)

    # Mostramos los resultados
    print(
        palabra,
        "-> P(Xi=1 | spam) =", round(p_spam, 4),
        "| P(Xi=1 | normal) =", round(p_normal, 4)
    )

dinero -> P(Xi=1 | spam) = 0.8 | P(Xi=1 | normal) = 0.0
gratis -> P(Xi=1 | spam) = 0.8 | P(Xi=1 | normal) = 0.0
premio -> P(Xi=1 | spam) = 0.4 | P(Xi=1 | normal) = 0.1667
proyecto -> P(Xi=1 | spam) = 0.0 | P(Xi=1 | normal) = 0.6667
reunion -> P(Xi=1 | spam) = 0.2 | P(Xi=1 | normal) = 0.5


### ¿Qué ocurre si una palabra nunca aparece en una clase?

Si una palabra nunca aparece en una clase, su **probabilidad condicional será igual a cero**.

Por ejemplo, la palabra **`proyecto`** nunca aparece en los correos clasificados como *spam*. Por lo tanto:

$$
P(X_{\text{proyecto}}=1 \mid Y=\text{spam})=\frac{0}{5}=0
$$

Esto produce el llamado **problema de frecuencia cero**. En Naive Bayes, las probabilidades condicionales se multiplican para calcular $P(X\mid Y)$, por lo que una sola probabilidad igual a cero puede hacer que todo el producto sea cero.

Para evitar este problema se utiliza normalmente el **suavizado de Laplace**, que permite asignar una pequeña probabilidad a eventos que no aparecieron en los datos de entrenamiento.

### Problema de frecuencia cero

Por la **hipótesis de independencia condicional** de Naive Bayes:

$$
X_i \perp X_j \mid Y
$$

podemos expresar la probabilidad conjunta de las características de un correo como el producto de las probabilidades individuales:

$$
P(X\mid Y)
=
P(X_1\mid Y)
P(X_2\mid Y)
\cdots
P(X_5\mid Y).
$$

Por ejemplo, supongamos que para una determinada clase obtenemos las siguientes probabilidades:

$$
0.8 \times 0.8 \times 0.4 \times 0 \times 0.2.
$$

Como una de las probabilidades es igual a cero, no importa cuánto valgan las demás:

$$
\boxed{P(X\mid Y)=0}.
$$

Por lo tanto, **una sola probabilidad condicional igual a cero anula todo el producto**.

Este fenómeno se conoce como **problema de frecuencia cero** y ocurre cuando una palabra nunca aparece en una determinada clase dentro de los datos de entrenamiento.

Para evitar este problema se puede utilizar el **suavizado de Laplace**, que asigna una pequeña probabilidad a los eventos que no fueron observados en el conjunto de entrenamiento.

## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [14]:
# TODO: calcula probabilidades condicionales con Laplace
# ============================================================
# PARTE 4: Suavizado de Laplace
# ============================================================

prob_spam_laplace = []
prob_normal_laplace = []

for i, palabra in enumerate(vocabulario):

    # Número de correos de cada clase que contienen la palabra
    conteo_spam = sum(vector[i] for vector in spam_vectores)
    conteo_normal = sum(vector[i] for vector in normal_vectores)

    # Número total de correos por clase
    N_spam = len(spam_vectores)
    N_normal = len(normal_vectores)

    # Suavizado de Laplace para Bernoulli
    p_spam = (conteo_spam + 1) / (N_spam + 2)
    p_normal = (conteo_normal + 1) / (N_normal + 2)

    # Guardamos resultados
    prob_spam_laplace.append(p_spam)
    prob_normal_laplace.append(p_normal)

    print(
        palabra,
        "-> P(Xi=1 | spam) =", round(p_spam, 4),
        "| P(Xi=1 | normal) =", round(p_normal, 4)
    )





dinero -> P(Xi=1 | spam) = 0.7143 | P(Xi=1 | normal) = 0.125
gratis -> P(Xi=1 | spam) = 0.7143 | P(Xi=1 | normal) = 0.125
premio -> P(Xi=1 | spam) = 0.4286 | P(Xi=1 | normal) = 0.25
proyecto -> P(Xi=1 | spam) = 0.1429 | P(Xi=1 | normal) = 0.625
reunion -> P(Xi=1 | spam) = 0.2857 | P(Xi=1 | normal) = 0.5


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [16]:
# TODO: como computar likelihood, score conjunto y posterior para "dinero gratis"
# ============================================================
# PARTE 5: Clasificar el correo "dinero gratis"
# ============================================================

correo_nuevo = "dinero gratis"

# Vectorizamos el correo
x = vectorizar(correo_nuevo, vocabulario)

print("Correo:", correo_nuevo)
print("Vector:", x)


# ------------------------------------------------------------
# 1. Calcular likelihood P(x | Y)
# ------------------------------------------------------------

likelihood_spam = 1
likelihood_normal = 1

for i, xi in enumerate(x):

    if xi == 1:
        # La palabra está presente
        likelihood_spam *= prob_spam_laplace[i]
        likelihood_normal *= prob_normal_laplace[i]

    else:
        # La palabra está ausente
        likelihood_spam *= (1 - prob_spam_laplace[i])
        likelihood_normal *= (1 - prob_normal_laplace[i])


print("P(x | spam) =", likelihood_spam)
print("P(x | normal) =", likelihood_normal)


# ------------------------------------------------------------
# 2. Calcular scores conjuntos
# ------------------------------------------------------------

score_spam = P_spam * likelihood_spam
score_normal = P_normal * likelihood_normal

print("Score spam =", score_spam)
print("Score normal =", score_normal)


# ------------------------------------------------------------
# 3. Calcular probabilidades posteriores
# ------------------------------------------------------------

normalizacion = score_spam + score_normal

posterior_spam = score_spam / normalizacion
posterior_normal = score_normal / normalizacion

print("P(spam | x) =", posterior_spam)
print("P(normal | x) =", posterior_normal)


# ------------------------------------------------------------
# 4. Decisión final
# ------------------------------------------------------------

if posterior_spam > posterior_normal:
    clase = "spam"
else:
    clase = "normal"

print("Clase predicha:", clase)

# claramente tenemos que clase p(normal|x)=1.46 % es muy poco
# y tenemos que para la clase p(spam|x)=98.54 % por lo tanto la prediccion es spam


Correo: dinero gratis
Vector: [1, 1, 0, 0, 0]
P(x | spam) = 0.17849705479859584
P(x | normal) = 0.002197265625
Score spam = 0.08113502490845266
Score normal = 0.0011985085227272725
P(spam | x) = 0.9854432517009721
P(normal | x) = 0.014556748299027745
Clase predicha: spam


## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

## Parte 6 — Interpretación

### 1. ¿Dónde se usa la hipótesis de independencia condicional?

La hipótesis de **independencia condicional** se utiliza cuando descomponemos la probabilidad conjunta de las características como el producto de las probabilidades de cada característica:

$$
P(X_1,X_2,\ldots,X_5 \mid Y)
=
\prod_{i=1}^{5} P(X_i \mid Y).
$$

Esto significa que, **una vez conocida la clase $Y$**, asumimos que la presencia o ausencia de cada palabra puede tratarse independientemente de las demás.

Por ejemplo, para el correo `"dinero gratis"`, cuyo vector es:

$$
x=(1,1,0,0,0),
$$

calculamos:

$$
P(x\mid Y)
=
P(X_1=1\mid Y)
P(X_2=1\mid Y)
P(X_3=0\mid Y)
P(X_4=0\mid Y)
P(X_5=0\mid Y).
$$

Esta multiplicación de probabilidades individuales es precisamente donde se utiliza la hipótesis de independencia condicional de **Naive Bayes**.

---

### 2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los $2^5$ vectores posibles?

Cada una de las cinco características es una variable binaria:

$$
X_i\in\{0,1\}.
$$

Como tenemos cinco palabras, existen:

$$
2^5=32
$$

vectores posibles, desde:

$$
(0,0,0,0,0)
$$

hasta:

$$
(1,1,1,1,1).
$$

Sin la hipótesis de independencia condicional sería necesario estimar la probabilidad conjunta de todas estas combinaciones para cada clase.

Naive Bayes simplifica el problema utilizando:

$$
P(X\mid Y)
=
\prod_i P(X_i\mid Y).
$$

Por lo tanto, solo necesitamos almacenar las probabilidades individuales:

$$
P(X_i=1\mid Y).
$$

Además, como cada $X_i$ es una variable Bernoulli:

$$
P(X_i=0\mid Y)
=
1-P(X_i=1\mid Y).
$$

De esta manera, podemos calcular la probabilidad aproximada de cualquier vector $X$ utilizando únicamente las probabilidades individuales de las cinco palabras.

Por esta razón, la hipótesis de independencia condicional **reduce considerablemente la complejidad del modelo**.

---

### 3. ¿Por qué Naive Bayes se considera un modelo generativo aunque aquí lo usemos para clasificar?

Naive Bayes se considera un **modelo generativo** porque aprende la distribución conjunta de las características y la clase:

$$
P(X,Y).
$$

Esta distribución puede escribirse como:

$$
P(X,Y)=P(Y)P(X\mid Y).
$$

En este laboratorio se estimaron precisamente estas dos componentes:

- El **prior** de cada clase:

$$
P(Y).
$$

- La distribución de las características condicionada a cada clase:

$$
P(X\mid Y).
$$

Una vez conocidas estas probabilidades, podemos aplicar el teorema de Bayes para calcular:

$$
P(Y\mid X)
=
\frac{P(Y)P(X\mid Y)}{P(X)}.
$$

Esta última probabilidad es la que utilizamos para realizar la clasificación.

El modelo se denomina **generativo** porque describe un posible proceso de generación de los datos: primero se selecciona una clase $Y$ de acuerdo con $P(Y)$ y, posteriormente, se generan las características $X$ según $P(X\mid Y)$.

Por lo tanto, aunque finalmente utilizamos:

$$
P(Y\mid X)
$$

para decidir si un correo es **spam** o **normal**, el modelo se construye aprendiendo:

$$
\boxed{P(Y)\quad\text{y}\quad P(X\mid Y)}.
$$

Esta es la razón fundamental por la cual **Naive Bayes es un modelo generativo utilizado para realizar tareas de clasificación**.